In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-transpiler
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Treceri de Transpiler AI
Trecerile de Transpiler alimentate de AI sunt treceri care funcționează ca înlocuitor direct al trecerilor Qiskit „tradiționale" pentru unele sarcini de transpilare. Acestea produc adesea rezultate mai bune decât algoritmii heuristici existenți (cum ar fi adâncime mai mică și număr mai mic de CNOT), dar sunt și mult mai rapide decât algoritmii de optimizare precum rezolvatorii de satisfiabilitate booleană. Trecerile de Transpiler AI pot rula în mediul tău local sau în cloud folosind Qiskit Transpiler Service, dacă faci parte din Planul Premium IBM Quantum&reg;, Planul Flex sau Planul On-Prem (prin IBM Quantum Platform API).

> **Note:** Trecerile de Transpiler alimentate de AI se află în stadiu de lansare beta și pot fi modificate.
>     Dacă ai feedback sau dorești să contactezi echipa de dezvoltare, te rugăm să folosești acest [canal Qiskit Slack Workspace](https://qiskit.slack.com/archives/C06KF8YHUAU).

Următoarele treceri sunt disponibile în prezent:

**Treceri de rutare**
 - `AIRouting`: Selectarea layout-ului și rutarea Circuit-ului

**Treceri de sinteză a Circuit-ului**
 - `AICliffordSynthesis`: Sinteza Circuit-ului Clifford
 - `AILinearFunctionSynthesis`: Sinteza Circuit-ului de funcție liniară
 - `AIPermutationSynthesis`: Sinteza Circuit-ului de permutare
 - `AIPauliNetworkSynthesis`: Sinteza Circuit-ului de rețea Pauli

Pentru a utiliza trecerile de Transpiler AI, mai întâi [instalează pachetul `qiskit-ibm-transpiler`](/guides/qiskit-transpiler-service#install-transpiler-service). Vizitează [documentația API qiskit-ibm-transpiler](https://docs.quantum.ibm.com/api/qiskit-ibm-transpiler) pentru mai multe informații despre diferitele opțiuni disponibile.

## Rulează trecerile de Transpiler AI local sau în cloud
Dacă vrei să folosești trecerile de Transpiler alimentate de AI în mediul tău local gratuit, instalează `qiskit-ibm-transpiler` cu câteva dependențe suplimentare, astfel:

In [1]:
from qiskit.transpiler import PassManager
from qiskit.circuit.library import efficient_su2
from qiskit_ibm_transpiler.ai.routing import AIRouting
from qiskit_ibm_runtime import QiskitRuntimeService

backend = QiskitRuntimeService().backend("ibm_torino")
ai_passmanager = PassManager(
    [
        AIRouting(
            backend=backend,
            optimization_level=2,
            layout_mode="optimize",
            local_mode=True,
        )
    ]
)


circuit = efficient_su2(101, entanglement="circular", reps=1)

transpiled_circuit = ai_passmanager.run(circuit)

Fără aceste dependențe suplimentare, trecerile de Transpiler alimentate de AI rulează în cloud prin Qiskit Transpiler Service (disponibil doar pentru utilizatorii Planului Premium IBM Quantum, Planului Flex sau Planului On-Prem (prin IBM Quantum Platform API)). După instalarea dependențelor suplimentare, modul implicit de rulare a trecerilor de Transpiler alimentate de AI este să folosești mașina ta locală.

## Trecerea de rutare AI
Trecerea `AIRouting` acționează atât ca etapă de layout, cât și ca etapă de rutare. Poate fi folosită într-un `PassManager` astfel:

In [ ]:
from qiskit.transpiler import PassManager

from qiskit_ibm_transpiler.ai.routing import AIRouting
from qiskit_ibm_transpiler.ai.synthesis import AILinearFunctionSynthesis
from qiskit_ibm_transpiler.ai.collection import CollectLinearFunctions
from qiskit_ibm_transpiler.ai.synthesis import AIPauliNetworkSynthesis
from qiskit_ibm_transpiler.ai.collection import CollectPauliNetworks
from qiskit.circuit.library import efficient_su2

ibm_torino = QiskitRuntimeService().backend("ibm_torino")
ai_passmanager = PassManager(
    [
        AIRouting(
            backend=ibm_torino,
            optimization_level=3,
            layout_mode="optimize",
            local_mode=True,
        ),  # Route circuit
        CollectLinearFunctions(),  # Collect Linear Function blocks
        AILinearFunctionSynthesis(
            backend=ibm_torino, local_mode=True
        ),  # Re-synthesize Linear Function blocks
        CollectPauliNetworks(),  # Collect Pauli Networks blocks
        AIPauliNetworkSynthesis(
            backend=ibm_torino, local_mode=True
        ),  # Re-synthesize Pauli Network blocks.
    ]
)

circuit = efficient_su2(10, entanglement="full", reps=1)

transpiled_circuit = ai_passmanager.run(circuit)

Aici, `backend` determină harta de cuplare pentru care se face rutarea, `optimization_level` (1, 2 sau 3) determină efortul de calcul depus în proces (o valoare mai mare produce de obicei rezultate mai bune, dar durează mai mult), iar `layout_mode` specifică modul de gestionare a selecției layout-ului.
`layout_mode` include următoarele opțiuni:

- `keep`: Aceasta respectă layout-ul setat de trecerile anterioare ale Transpiler-ului (sau folosește layout-ul trivial dacă nu este setat). Este utilizat de obicei doar atunci când Circuit-ul trebuie rulat pe Qubiți specifici ai dispozitivului. Produce adesea rezultate mai slabe deoarece are mai puțin spațiu pentru optimizare.
- `improve`: Aceasta folosește layout-ul setat de trecerile anterioare ale Transpiler-ului ca punct de plecare. Este utilă atunci când ai o estimare inițială bună pentru layout; de exemplu, pentru Circuit-uri construite într-un mod care urmează aproximativ harta de cuplare a dispozitivului. Este de asemenea utilă dacă vrei să încerci alte treceri de layout specifice combinate cu trecerea `AIRouting`.
- `optimize`: Acesta este modul implicit. Funcționează cel mai bine pentru Circuit-uri generale unde s-ar putea să nu ai estimări bune de layout. Acest mod ignoră selecțiile anterioare de layout.
- `local_mode`: Acest indicator determină unde rulează trecerea `AIRouting`. Dacă este `False`, `AIRouting` rulează de la distanță prin Qiskit Transpiler Service. Dacă este `True`, pachetul încearcă să ruleze trecerea în mediul tău local, cu o rezervă la modul cloud dacă dependențele necesare nu sunt găsite.

## Treceri de sinteză a Circuit-ului AI
Trecerile de sinteză a Circuit-ului AI îți permit să optimizezi bucăți din diferite tipuri de Circuit-uri ([Clifford](https://docs.quantum.ibm.com/api/qiskit/qiskit.quantum_info.Clifford), [Funcție Liniară](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.LinearFunction), [Permutare](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.Permutation#permutation), Rețea Pauli) prin re-sintetizarea lor. Un mod tipic de a folosi trecerea de sinteză este următorul:

In [ ]:
from qiskit_ibm_transpiler import generate_ai_pass_manager
from qiskit.circuit.library import efficient_su2
from qiskit_ibm_runtime import QiskitRuntimeService


backend = QiskitRuntimeService().backend("ibm_torino")
torino_coupling_map = backend.coupling_map


su2_circuit = efficient_su2(101, entanglement="circular", reps=1)

ai_transpiler_pass_manager = generate_ai_pass_manager(
    coupling_map=torino_coupling_map,
    ai_optimization_level=3,
    optimization_level=3,
    ai_layout_mode="optimize",
)

ai_su2_transpiled_circuit = ai_transpiler_pass_manager.run(su2_circuit)

Sinteza respectă harta de cuplare a dispozitivului: poate fi rulată în siguranță după alte treceri de rutare fără a perturba Circuit-ul, deci Circuit-ul general va respecta în continuare restricțiile dispozitivului. În mod implicit, sinteza va înlocui sub-Circuit-ul original doar dacă sub-Circuit-ul sintetizat îmbunătățește originalul (verificând în prezent doar numărul de CNOT), dar aceasta poate fi forțată să înlocuiască întotdeauna Circuit-ul prin setarea `replace_only_if_better=False`.

Următoarele treceri de sinteză sunt disponibile din `qiskit_ibm_transpiler.ai.synthesis`:

- *AICliffordSynthesis*: Sinteză pentru Circuit-uri [Clifford](https://docs.quantum.ibm.com/api/qiskit/qiskit.quantum_info.Clifford) (blocuri de Gate-uri `H`, `S` și `CX`). În prezent până la blocuri de nouă Qubiți.
- *AILinearFunctionSynthesis*: Sinteză pentru Circuit-uri de [Funcție Liniară](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.LinearFunction) (blocuri de Gate-uri `CX` și `SWAP`). În prezent până la blocuri de nouă Qubiți.
- *AIPermutationSynthesis*: Sinteză pentru Circuit-uri de [Permutare](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.Permutation#permutation) (blocuri de Gate-uri `SWAP`). Disponibil în prezent pentru blocuri de 65, 33 și 27 de Qubiți.
- *AIPauliNetworkSynthesis*: Sinteză pentru Circuit-uri de rețea Pauli (blocuri de Gate-uri `H`, `S`, `SX`, `CX`, `RX`, `RY` și `RZ`). În prezent până la blocuri de șase Qubiți.

Ne așteptăm să creștem treptat dimensiunea blocurilor suportate.

Toate trecerile folosesc un pool de fire de execuție pentru a trimite mai multe cereri în paralel. În mod implicit, numărul maxim de fire de execuție este numărul de nuclee plus patru (valorile implicite pentru obiectul Python `ThreadPoolExecutor`). Cu toate acestea, poți seta propria valoare cu argumentul `max_threads` la instanțierea trecerii. De exemplu, următoarea linie instanțiază trecerea `AILinearFunctionSynthesis`, permițându-i să utilizeze un maxim de 20 de fire de execuție.